# TD4 — Tabular Q-learning — Support

**Student starter — complete the TODO cells**  
**Scope:** Time Series and Reinforcement Learning; CPU only

## Working and claim contract

Local Python 3.11/3.12 with the locked `uv` environment is the primary path. Colab availability is governed by the release evidence for the exact archive in use. Public base seeds are `11, 23, 37, 51, 71`. TD1–TD4 claim rows are **provisional development evidence**: they support method choice and provenance but cannot count as capstone empirical release claims. Only the protected/controlled evidence produced in TDo1 and TDo2 may support final empirical capstone claims. Restart and run top-to-bottom before handoff.

## Support route

Guidance reduces syntax and navigation load; it does not expose the assessed solution or lower the Core standard.

In [ ]:
from pathlib import Path
import os, sys

def locate_course_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate/'uv.lock').is_file() and (candidate/'shared').is_dir(): return candidate.resolve()
    for lock in start.glob('*/uv.lock'):
        if (lock.parent/'shared').is_dir(): return lock.parent.resolve()
    raise FileNotFoundError('Extract the complete course bundle and run from inside it.')
COURSE_ROOT=locate_course_root(); os.chdir(COURSE_ROOT)
if str(COURSE_ROOT) not in sys.path: sys.path.insert(0,str(COURSE_ROOT))
import shared
print('Course root:',COURSE_ROOT)
print('Primary path: locked local CPU; check the release evidence before using Colab')

## Micro-example — exploration and truncation

The declared epsilon path is visible before training. Time-limit truncation still bootstraps. Core trains one Q-table through five seed blocks: this is one training run, not five independent training replicates. Epsilon decays once across all 1,750 episodes and the action RNG is initialized once from the first public seed.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from shared.rl import q_learning_target
assert q_learning_target(-1,5,.9,terminated=True)==-1
assert np.isclose(q_learning_target(-1,5,.9,terminated=False),3.5)
epsilon_curve=np.linspace(.85,.05,1750)
fig,ax=plt.subplots(figsize=(6,2.8)); ax.plot(epsilon_curve); ax.set(xlabel='global episode (five seed blocks)',ylabel='epsilon',title='Predeclared exploration schedule',ylim=(0,1)); plt.show()

## 1 — Restore MDP and separate seeds

In [ ]:
import hashlib, numpy as np, pandas as pd, matplotlib.pyplot as plt
from shared.data import PUBLIC_SEEDS
from shared.inventory_env import InventoryControlEnv
from shared.rl import q_learning_target, q_table_policy, random_policy, reorder_policy, evaluate_policies
assert q_learning_target(-1,5,.9,terminated=True)==-1
assert np.isclose(q_learning_target(-1,5,.9,terminated=False),3.5)
development_seeds=tuple(seed+100000 for seed in PUBLIC_SEEDS)
assert set(PUBLIC_SEEDS).isdisjoint(development_seeds)
print('training:',PUBLIC_SEEDS,'development:',development_seeds)

## 2 — Bounded explicit learning loop

The episode/step structure is supplied. You own the action, target, and update; incomplete code exits without fabricating evidence.

In [ ]:
def train_q_learning_student(*,seeds,episodes_per_seed=350,alpha=.18,gamma=.96,epsilon_start=.85,epsilon_end=.05):
    seeds=tuple(int(seed) for seed in seeds); assert seeds and episodes_per_seed>0
    q_table=np.zeros((9,2,3),float); rows=[]; global_episode=0
    total_episodes=len(seeds)*episodes_per_seed
    action_rng=np.random.default_rng(seeds[0])
    for seed in seeds:
        for local_episode in range(episodes_per_seed):
            epsilon=epsilon_start+(epsilon_end-epsilon_start)*(global_episode/max(1,total_episodes-1))
            env=InventoryControlEnv(); observation,_=env.reset(seed=seed*100000+local_episode)
            terminated=truncated=False; episode_cost=0.; action=target=None
            while not (terminated or truncated):
                state=tuple(int(v) for v in observation)
                # TODO 1: epsilon-greedy action using action_rng.
                action=None
                if action is None: break
                next_observation,reward,terminated,truncated,info=env.step(action)
                next_state=tuple(int(v) for v in next_observation)
                # TODO 2: target bootstraps on truncation; zero continuation only on termination.
                target=None
                if target is None: break
                # TODO 3: alpha-scaled in-place update q_table[state + (action,)].
                observation=next_observation; episode_cost+=float(info['cost'])
            env.close()
            if action is None or target is None: return None,None
            rows.append({'seed':seed,'episode':global_episode,'epsilon':epsilon,'episode_cost':episode_cost}); global_episode+=1
    training=pd.DataFrame(rows); assert len(training)==len(seeds)*episodes_per_seed and np.isfinite(q_table).all()
    return q_table,training
q_table,training=None,None # TODO: call after all three decisions are complete
print('Trainer pending' if q_table is None else q_table.shape)

## 3 — Intermediate assertions and explanatory diagnostics

In [ ]:
if q_table is not None:
    assert q_table.shape==(9,2,3) and np.isfinite(q_table).all() and len(training)==1750
    training.assign(rolling_cost=training.episode_cost.rolling(100,min_periods=20).mean()).plot(x='episode',y='rolling_cost',title='Training diagnostic — not independent evaluation')
    fig,ax=plt.subplots(figsize=(6,3)); image=ax.imshow(np.argmax(q_table,axis=2).T,aspect='auto',origin='lower'); ax.set(xlabel='stock',ylabel='regime',title='Greedy action index'); fig.colorbar(image,ax=ax); plt.show()
else: print('Diagnostics unlock after the assessed loop is complete.')

## 4 — Separate development replay and freeze

In [ ]:
development_results=pd.DataFrame()
if q_table is not None:
    policies={'random':random_policy,'reorder':reorder_policy,'q_learning':q_table_policy(q_table)}
    development_results=evaluate_policies(policies,seeds=development_seeds)
    assert len(development_results)==15 and development_results.groupby('policy').seed.nunique().eq(5).all()
    output=COURSE_ROOT/'02_labs/TD4_tabular_q_learning/q_table.npy'; np.save(output,q_table,allow_pickle=False)
    print('Q-table SHA256:',hashlib.sha256(output.read_bytes()).hexdigest())
development_results

## Required interpretation

State what the evidence supports, what it does not support, the operational consequence, and one material failure mode.

In [ ]:
# TODO: replace with 4–6 evidence-backed sentences.
interpretation=''
print(interpretation or 'Interpretation pending.')

## Optional Extension — ungraded

Freeze valid Core first. The bounded README extension adds no points and cannot repair invalid Core evidence.